# Topic 1: Spark Architecture & The Big Picture
## Subtopics: The Ecosystem: Spark Core, SQL, Streaming, MLlib, and GraphX.

## Cluster Components: Driver, Cluster Manager, and Executors.

The Execution Model: Jobs, Stages, and Tasks.

Lazy Evaluation & DAG: How Spark plans execution.

**Theory and Concepts**
Spark is a distributed computing engine. Unlike a traditional Python script that runs on one CPU, Spark breaks data into Partitions and processes them in parallel across a cluster.

**The Driver**: The "Brain." It runs your main() code and converts it into a logical plan called a DAG (Directed Acyclic Graph).

**Executors**: The "Muscles." These are JVM processes on worker nodes that execute the actual tasks and store data in memory (RAM).

**Lazy Evaluation**: Spark doesn't execute transformations (like filter or map) immediately. It just records them in the DAG. Execution only starts when an Action (like show() or count()) is called.

Spark decomposes your high-level Python commands into a three-tier structure: Jobs, Stages, and Tasks.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("SparkIntroductionSolutions") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
from pyspark.sql import SparkSession

# 1. Initialize SparkSession (The entry point to Spark)
spark = SparkSession.builder \
    .appName("IntroductionToSpark") \
    .master("local[*]") \
    .getOrCreate()

# 2. Check Spark Context details
print(f"Spark Version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")

# 3. Demonstrate Lazy Evaluation
data = [("Alice", 34), ("Bob", 45), ("Charlie", 28),("Alberto", 30)]
df = spark.createDataFrame(data, ["Name", "Age"])

# This line doesn't "run" the logic yet; it just updates the DAG
filtered_df = df.filter(df.Age > 40)

# This 'Action' triggers the actual computation
filtered_df.show()

In [ ]:
data = [("Alice", 34), ("Bob", 45), ("Charlie", 28),("Alberto", 30)]

In [ ]:
type(data)

In [ ]:
type(df)

In [ ]:
filtered_df.printSchema()

Problem 1: Create a SparkSession named "MyFirstApp" and create a DataFrame from a list of 5 numbers. Print the Spark application name.

Solution: spark.conf.get("spark.app.name") after initialization.



In [ ]:
# Create a simple list
numbers = [(1,), (2,), (3,), (4,), (5,)]
#numbers = [1, 2, 3, 4, 5]

# Create DataFrame
num_df = spark.createDataFrame(numbers, ["Column"])

# Accessing the configuration to prove the app name
app_name = spark.conf.get("spark.app.name")
print(f"The Application Name is: {app_name}")
num_df.show()

num_df.printSchema()

In [ ]:
type(numbers)

In [ ]:
numbers

In [ ]:
type(num_df)

Problem 2: Explain why calling df.filter() followed by df.select() doesn't produce an error even if the column name is misspelled until you call df.collect().

Solution: This is due to Lazy Evaluation. Spark only validates the full execution plan when an action is triggered.

In [ ]:
# Theoretical Demo:
# If you run:
bad_df = num_df.filter("WrongColumn > 10")
# Spark might not throw an error immediately in some deployment modes
# until you trigger an action like:
# bad_df.collect()
print("Reason: Spark uses a Lazy Evaluation model. It builds a Logical Plan (DAG) first.")
print("The plan is only optimized and validated against the actual data schema when an Action (like show/collect) is called.")

## Topic 2: DataFrames & Structured Operations
###Subtopics Data Sources: Reading CSV, JSON, and Parquet.

**Transformations**: select(), filter(), withColumn(), and drop().

**Schema Management**: Using StructType and StructField.

**Theory and Concepts**
A DataFrame is a distributed collection of data organized into named columns. It is conceptually similar to a Pandas DataFrame or a SQL table.

**Narrow Transformations**: Operations where each input partition contributes to only one output partition (e.g., filter, map). These are fast.

**Wide Transformations (Shuffles)**: Operations where data from multiple partitions is needed to compute the result (e.g., groupBy, join). These require moving data across the network, which is expensive.

In [ ]:
from pyspark.sql.functions import col, lit

# 1. Create Data
raw_data = [("Laptop", "Electronics", 1200), ("Shoes", "Apparel", 80), ("Phone", "Electronics", 600)]
df = spark.createDataFrame(raw_data, ["Item", "Category", "Price"])

# 2. Transformations
# Add a column for Tax (10%) and filter for high-price items
processed_df = df.withColumn("Tax", col("Price") * 0.1) \
                 .filter(col("Price") > 100) \
                 .select("Item", "Price", "Category","Tax")

processed_df_2 = df.withColumn("Tax", col("Price") * 0.1).filter(col("Price") > 100).select("Item", "Price", "Category","Tax")

df.show()

processed_df.show()

# 3. Print Schema
processed_df.printSchema()

In [ ]:
processed_df_2.show()

Problem 3: You have a DataFrame with columns FirstName and LastName. Create a new column FullName by concatenating them with a space.

Solution: from pyspark.sql.functions import concat_ws; df.withColumn("FullName", concat_ws(" ", col("FirstName"), col("LastName"))).



In [ ]:
# Sample Data
names_data = [("John", "Doe"), ("Jane", "Smith"), ("Alice", "Brown")]
names_df = spark.createDataFrame(names_data, ["FirstName", "LastName"])

# Solution using concat_ws (Concatenate With Separator)
names_df = names_df.withColumn("FullName", F.concat_ws(" ", F.col("FirstName"), F.col("LastName"))) \
                   .select("FullName")
names_df.show()

Problem 4: Filter a DataFrame to only include rows where the Price is between 50 and 500.

Solution: df.filter(col("Price").between(50, 500)).show().

In [ ]:
# Sample Data
products = [("Pen", 5), ("Headphones", 50), ("Monitor", 450), ("Car", 25000)]
products_df = spark.createDataFrame(products, ["Item", "Price"])

# Solution using .between()
filtered_products = products_df.filter(F.col("Price").between(51, 500))

filtered_products.show()

## Topic 3: Aggregations and Joins
###Subtopics Grouping: groupBy() and aggregate functions (sum, avg, count).

**Joins**: Inner, Left, Right, and Outer joins.

**Performance**: The impact of Shuffling.

**Theory and Concepts**
**Aggregations** in Spark involve a "Shuffle" phase. When you group by "Category," Spark must move all "Electronics" records to the same executor to sum them up.

**Joins**: Spark matches keys across two DataFrames. If one DataFrame is very small, Spark can perform a Broadcast Join, sending the small table to every executor to avoid a massive shuffle.

In [ ]:
from pyspark.sql.functions import avg, sum

# Dataset 1: Sales
sales_data = [("Store_A", 100), ("Store_B", 200), ("Store_A", 150),("Store_C", 1000)]
sales_df = spark.createDataFrame(sales_data, ["StoreID", "Amount"])

# Dataset 2: Store Locations
location_data = [("Store_A", "New York"), ("Store_B", "London"), ("Store_C", "Mexico"), ("Store_D", "Peru")]
loc_df = spark.createDataFrame(location_data, ["StoreID", "City"])

# 1. Aggregation: Average sales per store
avg_sales = sales_df.groupBy("StoreID").agg(sum("Amount").alias("AvgSales"))

# 2. Join: Combine sales with city names
final_df = avg_sales.join(loc_df, on="StoreID", how="inner")

final_df.show()

In [ ]:
sales_df.show()

In [ ]:
loc_df.show()

In [ ]:
final_df_2 = loc_df.join(avg_sales, on="StoreID", how="right")


In [ ]:
final_df_2.show()

In [ ]:
avg_sales.show()

Problemn 5: Given a DataFrame of Orders (OrderID, CustomerID, Total), find the total amount spent by each CustomerID.

Solution: df.groupBy("CustomerID").sum("Total").show().



In [ ]:
# Sample Data
orders_data = [(101, 1, 250.0), (102, 2, 100.0), (103, 1, 50.0), (104, 3, 300.0), (105, 3, 10000.0)]
orders_df = spark.createDataFrame(orders_data, ["OrderID", "CustomerID", "Total"])

# Solution using groupBy and sum
customer_spending = orders_df.groupBy("CustomerID") \
                             .agg(F.sum("Total").alias("TotalSpent"))

customer_spending.show()

Problem 6: Perform a left join between Employees and Departments on DeptID, and replace any null department names with "Unknown".

Solution: emp_df.join(dept_df, "DeptID", "left").fillna({"DeptName": "Unknown"}).

In [ ]:
# Sample Data
emp_data = [("Alice", 10), ("Bob", 20), ("Charlie", 30)] # Charlie has no valid Dept
dept_data = [(10, "HR"), (20, "Engineering")]

emp_df = spark.createDataFrame(emp_data, ["Name", "DeptID"])
dept_df = spark.createDataFrame(dept_data, ["DeptID", "DeptName"])

# 1. Join
joined_df = emp_df.join(dept_df, on="DeptID", how="inner")

# 2. Fill Nulls
#
final_join_df = joined_df.fillna({"DeptName": "Unknown"})

final_join_df.show()

In [ ]:
joined_df.show()

# The Scenario
You have been handed two datasets:

**Movies**: Contains MovieID, Title, and Genre.

**Ratings**: Contains UserID, MovieID, Rating (1–5), and Timestamp.

Goal: Identify the highest-rated genres that have a significant number of reviews.

1. Step 1: create data

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder.appName("MovieAnalyticsProject").getOrCreate()

# Mock Movie Data
movies_data = [
   (1, "Toy Story", "Animation|Children"),
   (2, "Jumanji", "Adventure|Children"),
   (3, "Heat", "Action|Crime"),
   (4, "Scream", "Horror"),
   (5, "The Matrix", "Action|Sci-Fi"),
   (6, "The Fellowship Of The Ring", "Action|Fantasy"),
   (7, "Shrek", "Animation|Comedy"),
   (8, "Flow", "Animation|Children"),
   (9, "Eternal Sunshine of the Spotless Minds", "Romance|Drama"),
   (10, "IT", "Terror"),
   (11, "2 Nacos en el Planeta de las Mujeres", "Action|Sci-Fi"),
   (12, "Rescatando al Soldado Perez", "Action|Comedy")
]


# Mock Ratings Data (User, MovieID, Rating)
ratings_data = [
   (101, 1, 4.0), (102, 1, 5.0), (103, 1, 4.5),
   (101, 2, 3.0), (102, 2, 2.0),
   (101, 3, 5.0), (103, 3, 4.0),
   (104, 4, 1.0),
   (101, 5, 5.0), (102, 5, 4.5), (104, 5, 5.0),
   (101, 6, 5.0),
   (102, 7, 5.0), (104, 7, 5.0),
   (101, 8, 5.0), (102, 8, 5.0), (103, 8, 5.0),
   (101, 10, 5.0), (102, 10, 3.0), (104, 10, 4.0),
   (101, 11, 2.0), (102, 11, 3.0), (104, 11, 1.0),
   (101, 12, 2.0), (102, 12, 3.0), (104, 12, 1.0)
]

movies_df = spark.createDataFrame(movies_data, ["MovieID", "Title", "Genres"])
ratings_df = spark.createDataFrame(ratings_data, ["UserID", "MovieID", "Rating"])

In [ ]:
movies_df.show()

In [ ]:
ratings_df.show()

Step 2: Data Cleaning & Transformation
Movies often belong to multiple genres (e.g., "Action|Sci-Fi"). We need to "explode" these so each genre gets its own row for analysis.

In [ ]:
# 1. Split the Genre string into an array and 'explode' it
# This turns one row ("The Matrix", "Action|Sci-Fi") into two rows.
exploded_movies = movies_df.withColumn("Genre", F.explode(F.split(F.col("Genres"), "\\|")))

exploded_movies.show()

# 2. Join the ratings with our exploded movies
# We use an inner join because a rating without a movie entry is useless for genre analysis.
movie_ratings_joined = ratings_df.join(exploded_movies, "MovieID")

movie_ratings_joined.select("Title", "Genre", "Rating").show(100)

Step 3: Analytical Aggregations
We want to calculate the Average Rating and the Total Number of Reviews for every genre.

In [ ]:
# Group by Genre and calculate metrics
genre_stats = movie_ratings_joined.groupBy("Genre").agg(
    F.avg("Rating").alias("AverageRating"),
    F.count("Rating").alias("ReviewCount")
)

# Filter for popularity: Only genres with more than 2 reviews
# This prevents a single 5-star review from skewing the results
popular_genres = genre_stats.filter(F.col("ReviewCount") >= 7) \
                            .orderBy(F.col("AverageRating").desc())

popular_genres.show()

Step 4: Final Insight & Export
Let's find the single best-performing movie (not just genre) that has an average rating above 4.0.

In [ ]:
# Complex Aggregation: Group by Title and find the best
top_movies = movie_ratings_joined.groupBy("Title") \
    .agg(F.round(F.avg("Rating"), 2).alias("FinalRating")) \
    .filter("FinalRating > 4.0") \
    .orderBy(F.col("FinalRating").desc())

top_movies.show()

# Theoretical Note: In production, you would save this to a Parquet file:
# top_movies.write.mode("overwrite").parquet("output/top_movies.parquet")

Problem extra: Find which UserID has given the most reviews, and what their average rating score is.

In [ ]:
user_stats = (
movie_ratings_joined
.groupBy("UserID")
.agg(
   F.avg("Rating").alias("AverageRating"),
   F.count("Rating").alias("ReviewCount")

)
.orderBy(F.col("ReviewCount").desc())
)
user_stats.show()

In [ ]:
ratings_df.show()

In [ ]:
user_analysis.show()

In [ ]:
from pyspark.sql import functions as F

# 1. Group by UserID
# 2. Calculate Count of ratings and Average rating per user
user_analysis = ratings_df.groupBy("UserID").agg(
    F.count("Rating").alias("ReviewCount"),
    F.round(F.avg("Rating"), 2).alias("AvgUserRating")
)

# 3. Sort by ReviewCount descending to find the most active user
#top_reviewers = user_analysis.orderBy(F.col("ReviewCount").desc())

top_reviewer = user_analysis.select(F.max("ReviewCount")).first()[0]

print("User Activity Report:")
print(top_reviewer)

##top_reviewers.show(1)


# 4. Filter for the single most active UserID
# top_user = top_reviewers.first()
# print(f"The most active user is {top_user['UserID']} with {top_user['ReviewCount']} reviews.")

**Theory and Concepts**
**Caching/Persistence**: If you plan to use the same DataFrame multiple times (like our movie_ratings_joined above), Spark will re-calculate it from the source every time an action is called. Caching stores the result in memory (RAM) for instant access.

**Partitioning**: Spark splits data into chunks (partitions). If you have 100 CPUs but only 2 partitions, 98 CPUs stay idle. Balancing partitions is key to performance.

In [ ]:
movie_ratings_joined.show()

In [ ]:
type(movie_ratings_joined)

In [ ]:
# rdd resilient distributed dataset

In [ ]:
# 1. Caching
# Use this when you will perform multiple actions on the same DF
movie_ratings_joined.cache()
movie_ratings_joined.count() # First time: calculates and stores in RAM
movie_ratings_joined.show()  # Second time: retrieves directly from RAM (much faster)

# 2. Checking Partitions
print(f"Current Partitions: {movie_ratings_joined.rdd.getNumPartitions()}")

# 3. Re-partitioning
# Increase partitions if your data is huge and you want more parallelism
optimized_df = movie_ratings_joined.repartition(10)
print(f"New Partition Count: {optimized_df.rdd.getNumPartitions()}")

In Spark, the first time you run an action, it reads from the source. Without caching, every subsequent action repeats that entire "read and calculate" cycle.

In this lesson, we will use Python's time library to measure the latency of these operations.

In [ ]:
import psutil
import os
from pyspark.sql import SparkSession

# 1. Physical Resource Check
logical_cores = psutil.cpu_count(logical=True)
physical_cores = psutil.cpu_count(logical=False)
total_ram_gb = round(psutil.virtual_memory().total / (1024**3), 2)

print(f"System Resources:")
print(f"- CPU Cores: {physical_cores} Physical, {logical_cores} Logical")
print(f"- Total RAM: {total_ram_gb} GB")

# 2. Spark-Specific Resource Check
# We initialize with 'local[*]' which tells Spark to use all available logical cores
spark = SparkSession.builder \
    .appName("AutoOptimizedSpark") \
    .master(f"local[{logical_cores}]") \
    .config("spark.driver.memory", f"{int(total_ram_gb * 0.5)}g") \
    .getOrCreate()

sc = spark.sparkContext
print(f"\nSpark Configuration:")
print(f"- Default Parallelism: {sc.defaultParallelism}")

In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("HighComplexityLab") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# 1. Increase Dataset to 100 Million Rows
# 2. Add "CPU-Melting" Complexity
# We repeat expensive operations multiple times to force CPU load
big_df = spark.range(0, 10_000_000) \
    .withColumn("val", F.rand()) \
    .withColumn("complex_calc",
        F.log(F.exp(F.sqrt(F.abs(F.sin(F.col("val")))))) +
        F.log(F.exp(F.sqrt(F.abs(F.cos(F.col("val")))))) +
        F.pow(F.col("val"), 5)
    )

# --- THE CACHING TEST ---
print("--- TEST 1: CACHING ---")
# First run: No cache
start = time.perf_counter()
big_df.agg(F.sum("complex_calc")).collect()
duration_no_cache = time.perf_counter() - start
print(f"Time (No Cache - Recomputing everything): {duration_no_cache:.2f}s")

# Second run: With cache
big_df.cache()
big_df.count() # Force the cache to fill
start = time.perf_counter()
big_df.agg(F.sum("complex_calc")).collect()
duration_with_cache = time.perf_counter() - start
print(f"Time (With Cache - Reading from RAM): {duration_with_cache:.2f}s")
print(f"Cache Speedup: {duration_no_cache / duration_with_cache:.1f}x\n")


# --- THE PARTITIONING TEST ---
print("--- TEST 2: PARTITIONING (Computation Only) ---")
# To avoid measuring the 'Shuffle cost', we prepare the DFs first
df_single = big_df.repartition(1).cache()
df_multi = big_df.repartition(2).cache()

# Warm up both
df_single.count()
df_multi.count()

# Measure 1 core working on 100M rows
start_1 = time.perf_counter()
df_single.agg(F.stddev("complex_calc")).collect()
duration_1 = time.perf_counter() - start_1
print(f"Time with 1 partition (Single Thread): {duration_1:.2f}s")

# Measure 8 cores working on 100M rows
start_8 = time.perf_counter()
df_multi.agg(F.stddev("complex_calc")).collect()
duration_8 = time.perf_counter() - start_8
print(f"Time with 2 partitions (Parallelized): {duration_8:.2f}s")

print(f"\nFinal Parallelism Win: {duration_1 / duration_8:.1f}x faster")